Clone Repo

In [ ]:
!git clone https://github.com/fashn-AI/fashn-vton-1.5.git
%cd fashn-vton-1.5

Cloning into 'fashn-vton-1.5'...
remote: Enumerating objects: 92, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 92 (delta 20), reused 13 (delta 13), pack-reused 67 (from 1)
Receiving objects: 100% (92/92), 334.44 KiB | 1.43 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/content/fashn-vton-1.5


Check Structure

In [ ]:
!find . -maxdepth 3 -type d

.
./.git
./.git/objects
./.git/objects/pack
./.git/objects/info
./.git/info
./.git/hooks
./.git/logs
./.git/logs/refs
./.git/refs
./.git/refs/heads
./.git/refs/tags
./.git/refs/remotes
./.git/branches
./examples
./examples/data
./tests
./src
./src/fashn_vton
./src/fashn_vton/preprocessing
./src/fashn_vton/__pycache__
./src/fashn_vton/dwpose
./src/fashn_vton/utils
./scripts


Install Dependencies

In [ ]:
!pip install -r requirements.txt

!pip install xformers accelerate
!pip install opencv-python pillow matplotlib
!pip install onnxruntime-gpu

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 4.5 MB/s eta 0:00:00


FIX PYTHON PATH (IMPORTANT)

In [ ]:
import sys
sys.path.append("/content/fashn-vton-1.5/src")

Test Import

In [ ]:
from fashn_vton import TryOnPipeline

print("Import Successful")

Import Successful


Download Weights

In [ ]:
!python scripts/download_weights.py --weights-dir ./weights



model.safetensors: 100% 1.94G/1.94G [00:12<00:00, 154MB/s]
  Saved to: /content/fashn-vton-1.5/weights/model.safetensors

yolox_l.onnx: 100% 217M/217M [00:01<00:00, 153MB/s]
  Saved to: /content/fashn-vton-1.5/weights/dwpose/yolox_l.onnx
dw-ll_ucoco_384.onnx: 100% 134M/134M [00:02<00:00, 66.6MB/s]
  Saved to: /content/fashn-vton-1.5/weights/dwpose/dw-ll_ucoco_384.onnx

config.json: 1.65kB [00:00, 4.27MB/s]
model.safetensors: 100% 256M/256M [00:04<00:00, 63.9MB/s]
Loading weights: 100% 930/930 [00:01<00:00, 706.06it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]
  Cached in HuggingFace hub cache

Download complete!

Weights directory structure:
    /content/fashn-vton-1.5/weights/
    ├── model.safetensors
    └── dwpose/
        ├── yolox_l.onnx
        └── dw-ll_ucoco_384.onnx

Usage:
    from fashn_vton import TryOnPipeline
    pipeline = TryOnPipeline(weights_dir="/content/fashn-vton-1.5/weights")



Upload Images

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving garment.jpg to garment (9).jpg
Saving person.webp to person (4).webp


Run Inference

In [ ]:
import gc
import torch
from PIL import Image

# Clear memory
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

# Load smaller images
person = Image.open("person.jpg").convert("RGB")
garment = Image.open("garment.jpg").convert("RGB")

person = person.resize((384,576))
garment = garment.resize((384,576))

# Run inference
result = pipeline(
    person_image=person,
    garment_image=garment,
    category="tops",
    garment_photo_type="flat-lay",
    num_samples=1,
    num_timesteps=15,
    guidance_scale=1.2,
    seed=None,
    segmentation_free=True
)

result.images[0].save("output.png")

print("DONE")

Show Output

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("output.png")

plt.figure(figsize=(8,12))
plt.imshow(img)
plt.axis("off")
plt.show()

Download Output

In [ ]:
files.download("output.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

recovery  cell

In [ ]:
%cd /content/fashn-vton-1.5

import sys
sys.path.insert(0, "/content/fashn-vton-1.5/src")

from fashn_vton import TryOnPipeline

pipeline = TryOnPipeline(
    weights_dir="./weights",
    device="cuda"
)

print("READY")

/content/fashn-vton-1.5


TryOnPipeline - INFO - Using device: cuda
TryOnPipeline - INFO - Using dtype: torch.bfloat16
TryOnPipeline - INFO - Loading TryOnModel from /content/fashn-vton-1.5/weights/model.safetensors
TryOnPipeline - INFO - TryOnModel loaded
TryOnPipeline - INFO - Loading DWPose from /content/fashn-vton-1.5/weights/dwpose
TryOnPipeline - INFO - DWPose loaded
TryOnPipeline - INFO - Loading FashnHumanParser
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/930 [00:00<?, ?it/s]

TryOnPipeline - INFO - FashnHumanParser loaded


READY


Clear GPU Memory

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 4            |        cudaMalloc retries: 5         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   2118 MiB |  11654 MiB |  18791 GiB |  18789 GiB |
|       from large pool |   2039 MiB |  11570 MiB |  18784 GiB |  18782 GiB |
|       from small pool |     78 MiB |    237 MiB |      7 GiB |      7 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   2118 MiB |  11654 MiB |  18791 GiB |  18789 GiB |
|       from large pool |   2039 MiB |  11570 MiB |  18784 GiB |